- ingest data from volume to delta table
- add metadata column such as file_name , ingest_timestamp

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
source_file_path = f"{landing_file_path}/sprints"
full_table_name = f"{catalog_name}.{bronze_schema}.sprint"

In [0]:
from pyspark.sql.types import StructField,StructType,DoubleType,StringType,IntegerType,DateType,LongType
# for production scenarion we define schema explicitly if new data is also come then also no issue occur, for development purpose we can used .option("inferSchema","true") this option
 
sprint_schema = StructType([
    StructField("constructorId", StringType()),
    StructField("date",DateType()),
    StructField("driverId", StringType()),
    StructField("grid", LongType()),
    StructField("laps", LongType()),
    StructField("number", LongType()),
    StructField("points", DoubleType()),
    StructField("position", LongType()),
    StructField("positionText", StringType()),
    StructField("raceName", StringType()),
    StructField("round", LongType()),
    StructField("season", LongType()),
    StructField("status", StringType()),
    StructField("url", StringType())
])


In [0]:
df_sprint = spark.read.format("json") \
            .option("header","true") \
            .schema(sprint_schema) \
            .option("multiline","true") \
            .load(source_file_path)


In [0]:
from pyspark.sql import functions as F

df_sprint_final = add_ingest_metadata(df_sprint)
display(df_sprint_final)


In [0]:
df_sprint_final.write.format("delta") \
                        .mode("overwrite") \
                        .saveAsTable(full_table_name)

In [0]:
display(spark.table(full_table_name))